# Laboratorio 7: NLP y embeddings

Pipeline reproducible para texto, SGNS, analogías y clasificación de noticias.

## Investigación de herramientas y conceptos

### `nn.Embedding`

Esta capa funciona como una tabla de consulta entrenable. `num_embeddings` define la cantidad de entradas del vocabulario. `embedding_dim` define la longitud de cada vector. `padding_idx` reserva una entrada cuyo gradiente no se actualiza. `sparse` permite gradientes dispersos y reduce trabajo cuando solo se consultan pocas filas.

### `nn.Embedding.from_pretrained`

Este método construye una tabla a partir de vectores existentes. El argumento `freeze` decide si los vectores permanecen fijos o reciben ajuste fino durante la tarea supervisada. Esta opción permite comparar representaciones congeladas y entrenables bajo el mismo clasificador.

### `nn.EmbeddingBag`

Esta capa combina consultas y agregación sin crear secuencias rellenadas. `offsets` indica dónde comienza cada documento dentro de un arreglo plano de identificadores. El modo `mean` calcula el promedio y es adecuado para representar noticias con distinta longitud. El modo `sum` conserva la magnitud asociada a la cantidad de tokens. El modo `max` conserva el valor máximo por dimensión.

### CBOW y skip gram

CBOW usa las palabras del contexto para predecir la palabra central. Suele ser rápido y estable para palabras frecuentes. Skip gram usa la palabra central para predecir palabras de contexto. Produce más ejemplos y suele representar mejor palabras menos frecuentes.

### Dos matrices en Word2Vec

Word2Vec mantiene una matriz de entrada `W` para palabras centrales y una matriz de salida `W prima` para palabras de contexto. Cada matriz aprende un papel distinto dentro de la tarea predictiva. Para usos posteriores normalmente se conserva `W`, aunque también es posible combinar ambas matrices si la evaluación lo justifica.

### Word2Vec y GloVe

Word2Vec aprende mediante una tarea predictiva local sobre pares de palabras. SGNS aproxima esta tarea con ejemplos positivos y negativos. GloVe parte de estadísticas globales de coocurrencia y ajusta vectores para representar razones entre esas frecuencias. La comparación debe considerar que GloVe usa un corpus mucho mayor y un procedimiento de entrenamiento diferente.

In [ ]:
from collections import Counter
from functools import partial
import json
import os
from pathlib import Path
import random
import re
import sys
sys.path.insert(0, str(Path.cwd() / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from embeddings_lab.analogies import analogy_3cosadd, analogy_3cosmul, answer_rank, evaluate_analogies, most_similar, tsne_projection
from embeddings_lab.classification import EmbeddingBagClassifier, embedding_bag_collate, encode_document, evaluate_classifier, stratified_split, train_classifier
from embeddings_lab.experiments import ExperimentConfig, count_model_parameters, count_skipgram_pairs, oov_rate, save_run_metadata, seed_everything
from embeddings_lab.preprocessing import build_vocabulary, iter_skipgram_pairs_from_sequences, normalize_text, subsample_ids, subsampling_keep_probabilities, word_tokenize
from embeddings_lab.sgns import SGNS, train_sgns


## Configuración reproducible

In [ ]:
RUN_FULL = os.getenv('EMBEDDINGS_LAB_FULL', '0') == '1'
config = ExperimentConfig()
seed_everything(config.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(config.output_dir).mkdir(parents=True, exist_ok=True)
print({'run_full': RUN_FULL, 'device': str(device), **config.to_dict()})


## Exploración del corpus

In [ ]:
if RUN_FULL:
    from datasets import load_dataset
    wiki_rows = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train', streaming=True, cache_dir='data/huggingface')
else:
    wiki_rows = [
        {'text': '= Language =\nThe king and queen rule the country.'},
        {'text': 'A woman and a man write computer programs.'},
        {'text': '= Science =\nGood models learn from repeated context.'},
    ] * 40

token_limit = config.corpus_token_limit if RUN_FULL else 2_000
processed_dir = Path('data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)
corpus_path = processed_dir / 'wikitext_subset.txt'
token_counts = Counter()
articles = 0
lines = 0
tokens = 0
header_pattern = re.compile(r'^=\s+[^=].*?\s+=$')
with corpus_path.open('w', encoding='utf-8') as corpus_file:
    for row in wiki_rows:
        for line in row['text'].splitlines() or [row['text']]:
            lines += 1
            articles += bool(header_pattern.fullmatch(line.strip()))
            sequence = word_tokenize(line)
            if not sequence:
                continue
            remaining = token_limit - tokens
            sequence = sequence[:remaining]
            corpus_file.write(' '.join(sequence) + '\n')
            token_counts.update(sequence)
            tokens += len(sequence)
            if tokens >= token_limit:
                break
        if tokens >= token_limit:
            break
corpus_summary = {'articles': articles, 'lines': lines, 'tokens': tokens, 'unique_tokens': len(token_counts)}
pd.DataFrame([corpus_summary])


## Tokenización

In [ ]:
from nltk.tokenize import TreebankWordTokenizer
library_tokenizer = TreebankWordTokenizer()
examples = [
    "I can't believe it's working.",
    'State-of-the-art systems change quickly.',
    'Dr. Smith arrived at 3.30 p.m.',
    'The price is 1,234.50 dollars.',
    'New York-based teams won.',
    'Email test@example.com today.',
    '<unk> is a reserved token.',
    'Children played, laughed and ran.',
    'Version 2.0-beta is available.',
    'France and Germany are countries.',
]
tokenizer_comparison = pd.DataFrame([
    {'text': text, 'custom': word_tokenize(text), 'library': library_tokenizer.tokenize(normalize_text(text))}
    for text in examples
])
tokenizer_comparison


## Ley de Zipf y vocabulario

In [ ]:
ranked_counts = np.asarray([count for _, count in token_counts.most_common()], dtype=np.int64)
ranks = np.arange(1, len(ranked_counts) + 1)
if len(ranked_counts):
    plt.figure(figsize=(7, 4))
    plt.loglog(ranks, ranked_counts)
    plt.xlabel('Rango')
    plt.ylabel('Frecuencia')
    plt.title('Frecuencia frente a rango')
    plt.show()
coverage_rows = []
for k in [10, 1_000, 30_000]:
    coverage_rows.append({'top_k': k, 'token_coverage': ranked_counts[:k].sum() / ranked_counts.sum() if ranked_counts.size else 0.0})
threshold_rows = []
for minimum in [2, 5, 10]:
    vocabulary_size = 1 + sum(count >= minimum for count in token_counts.values())
    known = sum(count for count in token_counts.values() if count >= minimum)
    threshold_rows.append({'min_count': minimum, 'vocabulary_size': vocabulary_size, 'oov_rate': 1 - known / tokens if tokens else 0.0})
display(pd.DataFrame(coverage_rows))
display(pd.DataFrame(threshold_rows))
active_min_count = config.min_count if RUN_FULL else 1
kept_tokens = sorted((token for token, count in token_counts.items() if count >= active_min_count), key=lambda token: (-token_counts[token], token))
vocabulary = {'<unk>': 0, **{token: index for index, token in enumerate(kept_tokens, 1)}}
id_to_token = [None] * len(vocabulary)
for token, token_id in vocabulary.items():
    id_to_token[token_id] = token
(processed_dir / 'vocabulary.json').write_text(json.dumps(vocabulary, ensure_ascii=False), encoding='utf-8')


## Submuestreo y pares skip gram

In [ ]:
with corpus_path.open(encoding='utf-8') as corpus_file:
    encoded_sequences = [[vocabulary.get(token, 0) for token in line.split()] for line in corpus_file]
keep_by_token = subsampling_keep_probabilities(token_counts, config.subsampling_threshold)
keep_by_id = [1.0] * len(vocabulary)
for token, token_id in vocabulary.items():
    keep_by_id[token_id] = keep_by_token.get(token, 1.0)
rng = random.Random(config.seed)
if RUN_FULL:
    subsampled_sequences = [list(subsample_ids(sequence, keep_by_id, rng=rng)) for sequence in encoded_sequences]
else:
    subsampled_sequences = encoded_sequences
discard_rows = []
for word in ['the', 'of', 'and']:
    token_id = vocabulary.get(word)
    original_count = sum(sequence.count(token_id) for sequence in encoded_sequences) if token_id is not None else 0
    retained_count = sum(sequence.count(token_id) for sequence in subsampled_sequences) if token_id is not None else 0
    observed = 100 * (original_count - retained_count) / original_count if original_count else 0.0
    discard_rows.append({'word': word, 'expected_discard_percent': 100 * (1 - keep_by_token.get(word, 1.0)), 'observed_discard_percent': observed})
pairs_before = sum(count_skipgram_pairs(sequence, config.window_size) for sequence in encoded_sequences)
pairs_after = sum(count_skipgram_pairs(sequence, config.window_size) for sequence in subsampled_sequences)
display(pd.DataFrame(discard_rows))
pair_counts = pd.DataFrame([{'pairs_before': pairs_before, 'pairs_after': pairs_after}])
pair_counts.to_csv(Path(config.output_dir) / 'pair_counts.csv', index=False)
del encoded_sequences
pair_counts


## Entrenamiento SGNS

In [ ]:
def pair_factory():
    return iter_skipgram_pairs_from_sequences(subsampled_sequences, config.window_size)

counts_by_id = np.zeros(len(vocabulary), dtype=np.float64)
for token, token_id in vocabulary.items():
    counts_by_id[token_id] = token_counts.get(token, 0)

wordsim_records = []
analogy_records = []
if RUN_FULL:
    from gensim.test.utils import datapath
    with open(datapath('wordsim353.tsv'), encoding='utf-8') as handle:
        for line in handle:
            if line.startswith('#'):
                continue
            left, right, score = line.rstrip().split('\t')
            wordsim_records.append((left.lower(), right.lower(), float(score)))
    category = None
    with open(datapath('questions-words.txt'), encoding='utf-8') as handle:
        for line in handle:
            if line.startswith(':'):
                category = line[1:].strip()
            else:
                analogy_records.append((category, *line.lower().split()))

semantic_categories = {'capital-common-countries', 'capital-world', 'currency', 'city-in-state', 'family'}
control_words = ['king', 'france', 'computer', 'good', 'january', 'run']
def epoch_evaluation(epoch, model, metrics):
    vectors = model.embeddings.detach().cpu().numpy()
    observed = []
    expected = []
    for left, right, score in wordsim_records:
        if left in vocabulary and right in vocabulary:
            first = vectors[vocabulary[left]]
            second = vectors[vocabulary[right]]
            denominator = np.linalg.norm(first) * np.linalg.norm(second)
            observed.append(float(first @ second / denominator) if denominator else 0.0)
            expected.append(score)
    semantic = [record[1:] for record in analogy_records if record[0] in semantic_categories]
    syntactic = [record[1:] for record in analogy_records if record[0] not in semantic_categories]
    semantic_metrics = evaluate_analogies(vectors, vocabulary, semantic) if semantic else {'accuracy': 0.0, 'evaluated': 0}
    syntactic_metrics = evaluate_analogies(vectors, vocabulary, syntactic) if syntactic else {'accuracy': 0.0, 'evaluated': 0}
    evaluated_total = semantic_metrics['evaluated'] + syntactic_metrics['evaluated']
    total_accuracy = (semantic_metrics['accuracy'] * semantic_metrics['evaluated'] + syntactic_metrics['accuracy'] * syntactic_metrics['evaluated']) / evaluated_total if evaluated_total else 0.0
    neighbors = {word: most_similar(vectors, vocabulary, word, top_k=5) for word in control_words if word in vocabulary}
    return {
        'wordsim_spearman': float(spearmanr(observed, expected).statistic) if len(observed) > 1 else 0.0,
        'semantic_accuracy': semantic_metrics['accuracy'],
        'syntactic_accuracy': syntactic_metrics['accuracy'],
        'total_accuracy': total_accuracy,
        'neighbors': neighbors,
    }

dimensions = config.dimensions if RUN_FULL else (8,)
epochs = config.epochs if RUN_FULL else 2
embedding_runs = {}
history_rows = []
for dimension in dimensions:
    model = SGNS(len(vocabulary), dimension, seed=config.seed)
    result = train_sgns(
        model, pair_factory, counts_by_id, epochs=epochs, batch_size=config.batch_size if RUN_FULL else 64,
        num_negatives=config.num_negatives, learning_rate=config.learning_rate, seed=config.seed,
        device=device, reserved_ids=(0,), epoch_callback=epoch_evaluation,
    )
    vectors = model.embeddings.detach().cpu().numpy()
    embedding_runs[dimension] = {'model': model, 'vectors': vectors, 'result': result}
    for row in result['epoch_records']:
        history_rows.append({'dimension': dimension, **row})
    run_dir = Path(config.output_dir) / f'sgns_dim_{dimension}'
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), run_dir / 'model.pt')
    np.save(run_dir / 'vectors.npy', vectors)
    save_run_metadata(config.output_dir, f'sgns_dim_{dimension}', {'dimension': dimension, 'tokens': tokens, 'epochs': epochs, 'min_count': active_min_count, 'total_time_seconds': result['total_time_seconds'], 'peak_gpu_memory_bytes': result['peak_gpu_memory_bytes']})
history = pd.DataFrame(history_rows)
history.to_csv(Path(config.output_dir) / 'sgns_history.csv', index=False)
run_summary = pd.DataFrame([{'dimension': dimension, 'total_time_seconds': run['result']['total_time_seconds'], 'peak_gpu_memory_bytes': run['result']['peak_gpu_memory_bytes'], **{key: run['result']['epoch_records'][-1].get(key, 0.0) for key in ['wordsim_spearman', 'semantic_accuracy', 'syntactic_accuracy', 'total_accuracy']}} for dimension, run in embedding_runs.items()])
run_summary['selection_score'] = run_summary['wordsim_spearman'].fillna(-1) + run_summary['total_accuracy'].fillna(0)
best_dimension = int(run_summary.sort_values(['selection_score', 'dimension'], ascending=[False, True]).iloc[0]['dimension'])
reference_dimension = 100 if 100 in embedding_runs else best_dimension
run_summary.to_csv(Path(config.output_dir) / 'sgns_summary.csv', index=False)
history


In [ ]:
if not history.empty:
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    for dimension, group in history.groupby('dimension'):
        axes[0].plot(group['epoch'], group['loss'], marker='o', label=str(dimension))
        axes[1].plot(group['epoch'], group['total_accuracy'], marker='o', label=str(dimension))
    axes[0].set_title('Pérdida por época')
    axes[1].set_title('Accuracy de analogías por época')
    for axis in axes:
        axis.set_xlabel('Época')
        axis.legend(title='Dimensión')
    plt.show()


## Analogías y visualización

In [ ]:
custom_analogies = [
    ('man', 'king', 'woman', 'queen', 'género'), ('boy', 'prince', 'girl', 'princess', 'género'), ('father', 'man', 'mother', 'woman', 'género'),
    ('france', 'paris', 'germany', 'berlin', 'capital'), ('italy', 'rome', 'japan', 'tokyo', 'capital'), ('spain', 'madrid', 'portugal', 'lisbon', 'capital'),
    ('france', 'french', 'germany', 'german', 'gentilicio'), ('italy', 'italian', 'spain', 'spanish', 'gentilicio'), ('japan', 'japanese', 'china', 'chinese', 'gentilicio'),
    ('good', 'better', 'bad', 'worse', 'comparativo'), ('good', 'best', 'bad', 'worst', 'superlativo'), ('small', 'smaller', 'large', 'larger', 'comparativo'),
    ('walk', 'walked', 'run', 'ran', 'verbo'), ('car', 'cars', 'book', 'books', 'plural'), ('child', 'children', 'man', 'men', 'plural'),
]
final_dimension = reference_dimension
final_vectors = embedding_runs[final_dimension]['vectors']
def analogia(a, b, c, k):
    return analogy_3cosadd(final_vectors, vocabulary, a, b, c, top_k=k)

from gensim.models import KeyedVectors
keyed_vectors = KeyedVectors(vector_size=final_dimension)
keyed_vectors.add_vectors(id_to_token, final_vectors)
vector_path = Path(config.output_dir) / 'final_sgns_vectors.txt'
keyed_vectors.save_word2vec_format(str(vector_path), binary=False)
analogy_rows = []
for a, b, c, expected, category in custom_analogies:
    missing = [word for word in [a, b, c, expected] if word not in vocabulary]
    if missing:
        analogy_rows.append({'category': category, 'query': f'{a}:{b}::{c}', 'expected': expected, 'status': 'oov', 'missing': missing})
        continue
    ranked = analogia(a, b, c, len(vocabulary))
    multiplied = analogy_3cosmul(final_vectors, vocabulary, a, b, c, top_k=5)
    unfiltered = analogy_3cosadd(final_vectors, vocabulary, a, b, c, top_k=5, exclude_inputs=False)
    va, vb, vc, vd = [final_vectors[vocabulary[word]] for word in [a, b, c, expected]]
    query = vb / np.linalg.norm(vb) - va / np.linalg.norm(va) + vc / np.linalg.norm(vc)
    answer_similarity = float(query @ vd / (np.linalg.norm(query) * np.linalg.norm(vd)))
    gensim_top = keyed_vectors.most_similar(positive=[b, c], negative=[a], topn=5)
    analogy_rows.append({'category': category, 'query': f'{a}:{b}::{c}', 'expected': expected, 'status': 'evaluated', 'top_5_3cosadd': ranked[:5], 'top_5_3cosmul': multiplied, 'gensim_top_5': gensim_top, 'same_top_1_as_gensim': ranked[0][0] == gensim_top[0][0], 'answer_rank': answer_rank(ranked, expected), 'answer_similarity': answer_similarity, 'without_exclusion': unfiltered})
analogy_table = pd.DataFrame(analogy_rows)
analogy_table.to_json(Path(config.output_dir) / 'individual_analogies.json', orient='records', force_ascii=False, indent=2)
analogy_table


In [ ]:
eligible_words = [word for word in id_to_token if word != '<unk>'][:500]
if len(eligible_words) >= 2:
    selected_vectors = np.asarray([final_vectors[vocabulary[word]] for word in eligible_words])
    coordinates = tsne_projection(selected_vectors, seed=config.seed)
    plt.figure(figsize=(12, 9))
    plt.scatter(coordinates[:, 0], coordinates[:, 1], s=8, alpha=0.6)
    for index in range(0, len(eligible_words), max(1, len(eligible_words) // 40)):
        plt.annotate(eligible_words[index], coordinates[index], fontsize=7)
    plt.title('Proyección t-SNE de palabras frecuentes')
    plt.show()


## Clasificación de AG News

In [ ]:
if RUN_FULL:
    news = load_dataset('fancyzhx/ag_news', cache_dir='data/huggingface')
    all_train_texts = list(news['train']['text'])
    all_train_labels = list(news['train']['label'])
    test_texts = list(news['test']['text'])
    test_labels = list(news['test']['label'])
else:
    templates = ['world leaders meet in city', 'team wins sports match', 'market reports company profit', 'new computer technology released']
    all_train_texts = [templates[label] + f' news {index}' for label in range(4) for index in range(20)]
    all_train_labels = [label for label in range(4) for _ in range(20)]
    test_texts = [templates[label] for label in range(4) for _ in range(4)]
    test_labels = [label for label in range(4) for _ in range(4)]
train_texts, validation_texts, train_labels, validation_labels = stratified_split(all_train_texts, all_train_labels, seed=config.seed)
split_path = processed_dir / 'ag_news_splits.jsonl'
with split_path.open('w', encoding='utf-8') as handle:
    for split_name, texts_part, labels_part in [('train', train_texts, train_labels), ('validation', validation_texts, validation_labels), ('test', test_texts, test_labels)]:
        for text, label in zip(texts_part, labels_part):
            handle.write(json.dumps({'split': split_name, 'text': text, 'label': label}, ensure_ascii=False) + '\n')
news_oov = oov_rate((word_tokenize(text) for text in all_train_texts), vocabulary)
def encoded_dataset(texts, labels):
    return [(encode_document(text, vocabulary), label) for text, label in zip(texts, labels)]
collate = partial(embedding_bag_collate, unknown_id=vocabulary['<unk>'])
train_loader = DataLoader(encoded_dataset(train_texts, train_labels), batch_size=128 if RUN_FULL else 16, shuffle=True, collate_fn=collate)
validation_loader = DataLoader(encoded_dataset(validation_texts, validation_labels), batch_size=256, shuffle=False, collate_fn=collate)
test_loader = DataLoader(encoded_dataset(test_texts, test_labels), batch_size=256, shuffle=False, collate_fn=collate)
news_oov


In [ ]:
classifier_epochs = 5 if RUN_FULL else 2
pretrained = torch.tensor(final_vectors, dtype=torch.float32)
variant_builders = {
    'random': lambda: EmbeddingBagClassifier(len(vocabulary), final_dimension, hidden_dim=128, dropout=0.2),
    'sgns_frozen': lambda: EmbeddingBagClassifier(len(vocabulary), final_dimension, pretrained_embeddings=pretrained, freeze=True, hidden_dim=128, dropout=0.2),
    'sgns_finetuned': lambda: EmbeddingBagClassifier(len(vocabulary), final_dimension, pretrained_embeddings=pretrained, freeze=False, hidden_dim=128, dropout=0.2),
}
classifier_rows = []
classifier_history_rows = []
trained_classifiers = {}
for name, builder in variant_builders.items():
    seed_everything(config.seed)
    classifier = builder()
    classifier_history = train_classifier(classifier, train_loader, validation_loader, epochs=classifier_epochs, device=device)
    validation_metrics = evaluate_classifier(classifier, validation_loader, device=device)
    classifier_rows.append({'variant': name, **count_model_parameters(classifier), **validation_metrics, 'training_seconds': sum(row['duration_seconds'] for row in classifier_history)})
    classifier_history_rows.extend({'variant': name, **row} for row in classifier_history)
    trained_classifiers[name] = {'model': classifier, 'history': classifier_history}
classification_results = pd.DataFrame(classifier_rows).sort_values('macro_f1', ascending=False)
classification_history = pd.DataFrame(classifier_history_rows)
classification_results.to_csv(Path(config.output_dir) / 'classification_validation.csv', index=False)
classification_history.to_csv(Path(config.output_dir) / 'classification_history.csv', index=False)
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, group in classification_history.groupby('variant'):
    axes[0].plot(group['epoch'], group['train_loss'], marker='o', label=name)
    axes[1].plot(group['epoch'], group['validation_loss'], marker='o', label=name)
axes[0].set_title('Pérdida de entrenamiento')
axes[1].set_title('Pérdida de validación')
for axis in axes:
    axis.set_xlabel('Época')
    axis.legend()
plt.show()
fraction_rows = []
fractions = [0.01, 0.25, 0.5, 1.0] if RUN_FULL else [1.0]
sgns_choice = classification_results[classification_results['variant'].str.startswith('sgns')].iloc[0]['variant']
for fraction in fractions:
    if fraction < 1.0:
        fraction_texts, _, fraction_labels, _ = train_test_split(train_texts, train_labels, train_size=fraction, stratify=train_labels, random_state=config.seed)
    else:
        fraction_texts, fraction_labels = train_texts, train_labels
    fraction_loader = DataLoader(encoded_dataset(fraction_texts, fraction_labels), batch_size=128 if RUN_FULL else 16, shuffle=True, collate_fn=collate)
    for name in ['random', sgns_choice]:
        seed_everything(config.seed)
        fraction_model = variant_builders[name]()
        fraction_history = train_classifier(fraction_model, fraction_loader, validation_loader, epochs=classifier_epochs, device=device)
        fraction_rows.append({'fraction': fraction, 'initialization': 'random' if name == 'random' else 'sgns', **count_model_parameters(fraction_model), 'training_seconds': sum(row['duration_seconds'] for row in fraction_history), 'history': fraction_history, **evaluate_classifier(fraction_model, validation_loader, device=device)})
fraction_results = pd.DataFrame(fraction_rows)
fraction_results.drop(columns=['history']).to_csv(Path(config.output_dir) / 'classification_fractions.csv', index=False)
(Path(config.output_dir) / 'classification_fraction_histories.json').write_text(json.dumps(fraction_rows, indent=2), encoding='utf-8')
for initialization, group in fraction_results.groupby('initialization'):
    plt.plot(group['fraction'], group['macro_f1'], marker='o', label=initialization)
plt.xlabel('Fracción de entrenamiento')
plt.ylabel('F1 macro de validación')
plt.legend()
plt.show()
classification_results


In [ ]:
def evaluate_test_once(model):
    targets = []
    predictions = []
    total_loss = 0.0
    model.train(False)
    with torch.no_grad():
        for token_ids, offsets, labels in test_loader:
            logits = model(token_ids.to(device), offsets.to(device))
            total_loss += float(torch.nn.functional.cross_entropy(logits, labels.to(device), reduction='sum'))
            targets.extend(labels.tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    precision, recall, f1, _ = precision_recall_fscore_support(targets, predictions, labels=[0, 1, 2, 3], average='macro', zero_division=0)
    metrics = {'loss': total_loss / len(targets), 'accuracy': accuracy_score(targets, predictions), 'macro_precision': precision, 'macro_recall': recall, 'macro_f1': f1, 'count': len(targets)}
    return metrics, confusion_matrix(targets, predictions, labels=[0, 1, 2, 3])

selected_variants = {'random': 'random', 'sgns': sgns_choice}
test_rows = []
test_matrices = {}
figure, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, (initialization, variant) in zip(axes, selected_variants.items()):
    metrics, matrix = evaluate_test_once(trained_classifiers[variant]['model'])
    test_rows.append({'initialization': initialization, 'variant': variant, **metrics})
    test_matrices[initialization] = matrix.tolist()
    ConfusionMatrixDisplay(matrix).plot(cmap='Blues', ax=axis, colorbar=False)
    axis.set_title(initialization)
plt.show()
test_results = pd.DataFrame(test_rows)
test_results.to_csv(Path(config.output_dir) / 'classification_test.csv', index=False)
(Path(config.output_dir) / 'confusion_matrices.json').write_text(json.dumps(test_matrices, indent=2), encoding='utf-8')
test_results
